# SWMS — Option B: class weights + fine-tuning MobileNetV2

Experiment from `ml/README.md` → *10. Next steps*. **It does not touch the served model**:
`models/swms_mobilenetv2_best.keras` is only read (as the baseline), and the result is a new file,
`swms_mobilenetv2_option_b.keras`.

All training logic lives in [`src/training/train_option_b.py`](../src/training/train_option_b.py), which reuses
`src/training/train.py` (same datasets, same model graph with `preprocess_input` inside it) and
`src/evaluation/evaluate.py` (same metrics files). This notebook only calls those functions step by step so the
outputs are visible.

**Fairness rules (same as for the current model):** same `dataset_split/` folder on Google Drive, model chosen by
**validation** loss only, test set evaluated **once** at the end.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all* (or run the cells in order).
If step 4 installs packages it restarts the session; after that, run all cells again.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime → Change runtime type → T4 GPU"

## 2. Google Drive and the dataset split

The current model was trained on `MyDrive/SWMS/dataset_split/` (made once by `scripts/split_dataset.py`).
Option B **must reuse that exact folder**: re-running the split can put different files in each split
(`os.listdir` order, see README section 1), and then the comparison with the current model is not fair.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DATASET_DIR = "/content/drive/MyDrive/SWMS/dataset_split"
assert os.path.isdir(DATASET_DIR), (
    f"{DATASET_DIR} not found. Use the Google account that ran the original notebook, "
    "or see the optional fallback cell below."
)
print("Dataset split found:", sorted(os.listdir(DATASET_DIR)))

*Optional fallback, only if `dataset_split/` is gone.* Uncomment and run it after step 3. It downloads Kaggle
version 8 and re-runs the split. The counts will match, but **which** files are in each split may not. Step 11
then reports *Baseline reproduced: False*, and the comparison is weaker: the current model may have been trained
on images that are now in the test set.

In [ ]:
# from google.colab import files
# files.upload()  # kaggle.json
# !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
# !pip install -q kaggle==2.0.2
# !kaggle datasets download sumn2u/garbage-classification-v2/8 -p /content/kaggle_raw --unzip
# !cd /content/repo/ml && python scripts/split_dataset.py

## 3. Get the code (this branch)

In [ ]:
BRANCH = "feat/ml-option-b"
!rm -rf /content/repo
!git clone --depth 1 --branch {BRANCH} https://github.com/jieffrey/Smart-Waste-System-Management.git /content/repo
%cd /content/repo/ml
!git log -1 --oneline

import sys
if "/content/repo/ml" not in sys.path:
    sys.path.insert(0, "/content/repo/ml")

## 4. Same TensorFlow / Keras as the API

The API loads models with TensorFlow 2.21.0 and Keras 3.13.2 (`requirements.txt`). Training with the same
versions keeps the saved `.keras` file loadable there. If Colab has other versions, this cell installs the
pinned ones and **restarts the session**; then run all cells again (this cell will just print *OK*).

In [ ]:
from importlib.metadata import version
import os

WANTED = {"tensorflow": "2.21.0", "keras": "3.13.2"}
have = {name: version(name) for name in WANTED}
print("Installed:", have)
if have != WANTED:
    !pip install -q tensorflow==2.21.0 keras==3.13.2
    print("Installed the pinned versions. Restarting the session: run all cells again afterwards.")
    os.kill(os.getpid(), 9)
else:
    print("OK")

## 5. Seeds and environment

In [ ]:
%matplotlib inline
import importlib
from src.training import train, train_option_b as ob
from src.evaluation import evaluate
importlib.reload(train); importlib.reload(evaluate); importlib.reload(ob)

seeds = ob.set_global_seeds()  # Python random, NumPy, TensorFlow (+ PYTHONHASHSEED); dataset shuffle seed 42 as before
env = ob.environment_info()
print("\nOutput folder on Drive:", ob.OUTPUT_ROOT)
print("Option B model file:   ", ob.MODEL_PATH)
print("Baseline (read only):  ", ob.CURRENT_MODEL_PATH)

## 6. Same split, same preprocessing

`check_split` stops the notebook if the per-class counts differ from the current model's split
(697/150/150, 1,175/252/253, 1,388/298/298) or if the 3 parity samples are not in `test/`.
The datasets come from `train.load_datasets`, which is the exact code of the current model:
`image_dataset_from_directory`, 224×224 bilinear, batch 32, training shuffle seed 42, no augmentation.
The model graph comes from `train.build_model`, with `preprocess_input` **inside** the model, so the API keeps
sending raw 0–255 pixels.

In [ ]:
ob.check_split(ob.DATASET_DIR)
train_ds, val_ds, raw_train_ds = train.load_datasets(ob.DATASET_DIR, return_raw=True)
class_names = raw_train_ds.class_names
assert class_names == ["organic", "paper", "plastic"], class_names

## 7. Class weights (train split only)

Balanced formula: `weight = total / (classes × count)`, from the **training** files only (validation and test
play no part). Expected from the README: `organic` ≈ 1.559, `paper` ≈ 0.925, `plastic` ≈ 0.783.
The weights only change the **training** loss; `val_loss` stays unweighted, so model selection is comparable
with the current model.

In [ ]:
class_weight = ob.compute_train_class_weights(raw_train_ds)
class_weight

## 8. Phase 1: frozen base, train the head

Same model and learning rate (Adam 0.001) as the current model, but with class weights and up to
**20 epochs** instead of 3. `EarlyStopping(val_loss, patience=3, restore_best_weights=True)` stops when
validation loss stops improving; `ModelCheckpoint` keeps the best epoch.

In [ ]:
model = train.build_model(learning_rate=ob.PHASE1_LEARNING_RATE)
phase1_params = ob.trainable_param_count(model)
print("Phase 1 trainable params:", phase1_params)  # 3,843: only the Dense head

h1 = ob.fit_phase(model, train_ds, val_ds, class_weight, ob.PHASE1_EPOCHS, ob.CHECKPOINT_DIR / "phase1_best.keras")
print("Phase 1 best:", ob.best_epoch(h1))

## 9. Phase 2: fine-tune the top of MobileNetV2

**How many layers, and why:** unfreeze from `block_13_expand` onward, the **top 38 of the 154** base layers
(index 116+).
- `block_13` is where MobileNetV2's **last stage** starts: it downsamples to 7×7 feature maps (stride 32), and
  blocks 13–16 plus `Conv_1` work at that resolution. These layers hold the most ImageNet-specific, high-level
  features (object parts and categories), which are the ones worth adapting to waste materials.
- The earlier stages learn edges, textures and simple shapes. These transfer well and stay frozen, which matters
  with only 3,260 training images (697 `organic`).
- Cutting at a stage boundary keeps every block in that stage trainable together. A cut in the middle of the
  stage (e.g. `block_15`) would split layers that work on the same feature maps.
- Unfreezing deeper (e.g. from `block_12`, layer 107) would add earlier-stage layers for little extra
  task-specific benefit and more overfitting risk.

**BatchNorm stays frozen:** the 13 BatchNorm layers in that range keep `trainable = False` and the base is still
called with `training=False`, so their ImageNet statistics don't get replaced by noisy batch-of-32 statistics.
That leaves **25 trainable layers** in the base (about 1.66 M trainable parameters with the head; printed below).

**Learning rate 1e-5** (100× lower than phase 1) so the pretrained weights move only slightly. Class weights and
early stopping as in phase 1. The phase-2 checkpoint is only written if it **beats phase 1's best validation
loss**. If fine-tuning doesn't help, the phase-1 model is kept (step 10).

In [ ]:
fine_tune = ob.unfreeze_top(model)  # recompiles with Adam(1e-5)
h2 = ob.fit_phase(
    model, train_ds, val_ds, class_weight, ob.PHASE2_EPOCHS, ob.CHECKPOINT_DIR / "phase2_best.keras",
    initial_value_threshold=ob.best_epoch(h1)["val_loss"],
)
print("Phase 2 best:", ob.best_epoch(h2))

## 10. Choose the final model (validation loss only) and save the curves

Saved as a **new** file `models/swms_mobilenetv2_option_b.keras` under the Option B folder on Drive.

In [ ]:
selection = ob.select_final_model(h1, h2)
ob.save_histories(h1, h2)  # training_history_phase1/2.csv + training_curves.png (both phases)
ob.write_run_config(
    seeds=seeds, environment=env,
    class_weight={class_names[k]: v for k, v in class_weight.items()},
    phase1_trainable_params=phase1_params, fine_tuning=fine_tune, selection=selection,
)

## 11. Test set: evaluated once

Nothing above looked at the test set. This cell:
1. writes the Option B predictions for the 3 parity samples (`metrics/expected_option_b.json`, same method as the
   original notebook cell 28, same format as `tests/samples/expected.json`);
2. evaluates **Option B once**: accuracy, loss, per-class precision/recall/F1, confusion matrix, misclassified
   grid;
3. evaluates the **current model** on the same test set as the baseline (read only, results in
   `results/option_b/current_model/`). Its confusion matrix must match the documented one
   (`[[145,3,2],[2,250,1],[0,8,290]]`); this proves it's the same test set and pipeline. Its misclassified grid
   is the one the README's error analysis (section 7) is missing.

**Do not re-run this cell after changing anything above.** Then the test set would be part of model selection.

In [ ]:
import keras
from IPython.display import Image as Show, display
from sklearn.metrics import classification_report

test_ds = evaluate.load_test_dataset(ob.DATASET_DIR)
assert test_ds.class_names == class_names

expected_b = ob.colab_parity_samples(keras.models.load_model(ob.MODEL_PATH))
evaluation = ob.final_evaluation(test_ds, class_names)

for label, key in [("Option B", "option_b"), ("Current model", "current")]:
    r = evaluation[key]
    print(f"\n=== {label}: accuracy {r['accuracy']:.4f}, loss {r['loss']:.4f} ===")
    print(classification_report(r["y_true"], r["y_pred"], target_names=class_names, digits=4))

for path in ["figures/confusion_matrix.png", "figures/misclassified.png",
             "current_model/figures/confusion_matrix.png", "current_model/figures/misclassified.png"]:
    print(path)
    display(Show(filename=str(ob.RESULTS_DIR / path)))

## 12. Current model vs Option B

In [ ]:
from IPython.display import Markdown
display(Markdown(ob.comparison_table(evaluation, class_names)))

## 13. Download the results

`option_b_artifacts.zip` contains `models/swms_mobilenetv2_option_b.keras` and `results/option_b/`.
Unzip it **inside `ml/`** of the repo. It adds new files only; nothing existing is overwritten.
A copy also stays on Drive (`MyDrive/SWMS/option_b_artifacts.zip`).

In [ ]:
archive = ob.make_zip()
from google.colab import files
files.download(str(archive))